In [13]:
import nltk
import numpy as np
import re

from nltk.corpus import movie_reviews
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding,SimpleRNN, LSTM, Dense

nltk.download('movie_reviews')

[nltk_data] Downloading package movie_reviews to /root/nltk_data...
[nltk_data]   Package movie_reviews is already up-to-date!


True

In [27]:
texts = [movie_reviews.raw(fileid)
         for fileid in movie_reviews.fileids()]

def clean_text(text):
    text = text.lower()
    text = re.sub(r"[^a-zA-Z]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

idx = np.arange(len(texts))
np.random.shuffle(idx)

texts = [clean_text(t) for t in texts]
texts = [texts[i] for i in idx][:1000]
max_words = 10000
tokenizer = Tokenizer(num_words=max_words, oov_token="<OOV>")
tokenizer.fit_on_texts(texts)

sequences = tokenizer.texts_to_sequences(texts)

input_sequences = []

for seq in sequences:
    for i in range(1, len(seq)):
        input_sequences.append(seq[:i])

max_len = 30

input_sequences = pad_sequences(input_sequences, maxlen=max_len, padding='pre')

X = input_sequences[:, :-1]
y = input_sequences[:, -1]

print(input_sequences)

[[   0    0    0 ...    0    0   31]
 [   0    0    0 ...    0   31  119]
 [   0    0    0 ...   31  119   19]
 ...
 [6428  543    2 ...    7  382  640]
 [ 543    2  492 ...  382  640    6]
 [   2  492 4595 ...  640    6 3055]]


In [5]:
len(texts)

1000

In [15]:
model = Sequential([
    Embedding(max_words, 128, input_length=max_len-1),
    SimpleRNN(128),
    Dense(256, activation='softmax'),
    Dense(max_words, activation='softmax')
])
model.summary()


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_1 (SimpleRNN)        │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [16]:
y = [y[i] for i in idx]

In [17]:
y = y[:1000]

In [18]:
len(y)

1000

In [21]:
def generate(pretext, next_words=20):
    for _ in range(next_words):
        seq = tokenizer.texts_to_sequences([pretext])[0]
        seq = pad_sequences([seq], maxlen=max_len-1, padding='pre')

        predicted = model.predict(seq)
        next_w_id = np.argmax(predicted)

        for word, index in tokenizer.word_index.items():
            if index == next_w_id:
                pretext += " " + word
                break

    return pretext

print(generate("this movie was", 20))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
this movie was expose expose expose expose penny expose expose expose penny penny expose expose remaining expose penny remaining remaining expose expose expose


In [22]:

model_lstm = Sequential([
    Embedding(max_words, 64, input_length=max_len-1),
    LSTM(64),
    Dense(max_words, activation='softmax')
])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [25]:
model_lstm.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

In [29]:
model.fit(X[:1000], y[:1000], epochs=5, batch_size=128)

Epoch 1/5
8/8 ━━━━━━━━━━━━━━━━━━━━ 4s 216ms/step - accuracy: 0.0100 - loss: 9.2053
Epoch 2/5
8/8 ━━━━━━━━━━━━━━━━━━━━ 2s 187ms/step - accuracy: 0.0460 - loss: 9.1860
Epoch 3/5
8/8 ━━━━━━━━━━━━━━━━━━━━ 2s 128ms/step - accuracy: 0.0570 - loss: 9.1649
Epoch 4/5
8/8 ━━━━━━━━━━━━━━━━━━━━ 1s 126ms/step - accuracy: 0.0570 - loss: 9.1401
Epoch 5/5
8/8 ━━━━━━━━━━━━━━━━━━━━ 1s 127ms/step - accuracy: 0.0530 - loss: 9.1115


In [30]:
def generate(pretext, next_words=20):
    for _ in range(next_words):
        seq = tokenizer.texts_to_sequences([pretext])[0]
        seq = pad_sequences([seq], maxlen=max_len-1, padding='pre')

        predicted = model_lstm.predict(seq)
        next_w_id = np.argmax(predicted)

        for word, index in tokenizer.word_index.items():
            if index == next_w_id:
                pretext += " " + word
                break

    return pretext

print(generate("this movie was", 20))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 314ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 71ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 120ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
this movie was the the the the the the the the the the the the the the the the the the the the
